In [3]:
import pandas as pd
import numpy as np

In [4]:
deliveries = pd.read_csv(r"C:\Users\dhruv\playXI\data\deliveries_updated_ipl_upto_2025.csv")
matches = pd.read_csv(r"C:\Users\dhruv\playXI\data\matches_updated_ipl_upto_2025.csv")
print("Data loaded successfully!")

Data loaded successfully!


In [6]:
print("Deliveries shape:", deliveries.shape)
print("Matches shape:", matches.shape)
print("Deliveries columns:", deliveries.columns.tolist())

Deliveries shape: (278205, 20)
Matches shape: (1169, 28)
Deliveries columns: ['matchId', 'inning', 'over_ball', 'over', 'ball', 'batting_team', 'bowling_team', 'batsman', 'non_striker', 'bowler', 'batsman_runs', 'extras', 'isWide', 'isNoBall', 'Byes', 'LegByes', 'Penalty', 'dismissal_kind', 'player_dismissed', 'date']


In [7]:
print("missing values in the deliveries:")
print(deliveries.isnull().sum())

missing values in the deliveries:
matchId                  0
inning                   0
over_ball                0
over                     0
ball                     0
batting_team             0
bowling_team             0
batsman                  0
non_striker              0
bowler                   0
batsman_runs             0
extras                   0
isWide              269125
isNoBall            277042
Byes                277504
LegByes             273990
Penalty             278203
dismissal_kind      264382
player_dismissed    264382
date                     0
dtype: int64


In [10]:
deliveries= deliveries.drop(columns=['Byes', 'LegByes', 'Penalty'])
print("droppped useless columns!")
print("new shape:", deliveries.shape)

droppped useless columns!
new shape: (278205, 17)


In [11]:
# Fill missing values
# isWide and isNoBall: NaN means it was a normal ball, so fill with 0
deliveries['isWide'] = deliveries['isWide'].fillna(0)
deliveries['isNoBall'] = deliveries['isNoBall'].fillna(0)

# dismissal_kind and player_dismissed: NaN means no wicket, fill with "none"
deliveries['dismissal_kind'] = deliveries['dismissal_kind'].fillna('none')
deliveries['player_dismissed'] = deliveries['player_dismissed'].fillna('none')

print("Missing values fixed!")
print(deliveries.isnull().sum())

Missing values fixed!
matchId             0
inning              0
over_ball           0
over                0
ball                0
batting_team        0
bowling_team        0
batsman             0
non_striker         0
bowler              0
batsman_runs        0
extras              0
isWide              0
isNoBall            0
dismissal_kind      0
player_dismissed    0
date                0
dtype: int64


In [12]:
# Feature Engineering
# Step 1: Calculate total runs scored by each batsman in each match
batsman_match_runs = deliveries.groupby(['matchId', 'batsman'])['batsman_runs'].sum().reset_index()

# Rename the column to something clear
batsman_match_runs.columns = ['matchId', 'batsman', 'total_runs']

print("Batsman match runs:")
print(batsman_match_runs.head(10))

Batsman match runs:
   matchId          batsman  total_runs
0   335982        AA Noffke           9
1   335982          B Akhil           0
2   335982      BB McCullum         158
3   335982         CL White           6
4   335982        DJ Hussey          12
5   335982        JH Kallis           8
6   335982       MV Boucher           7
7   335982  Mohammad Hafeez           5
8   335982          P Kumar          18
9   335982         R Dravid           2


In [13]:
# Step 2: Calculate balls faced by each batsman per match
# We only count legal balls (not wides)
batsman_balls = deliveries[deliveries['isWide'] == 0].groupby(
    ['matchId', 'batsman'])['ball'].count().reset_index()

batsman_balls.columns = ['matchId', 'batsman', 'balls_faced']

print("Balls faced:")
print(batsman_balls.head(10))

Balls faced:
   matchId          batsman  balls_faced
0   335982        AA Noffke           10
1   335982          B Akhil            2
2   335982      BB McCullum           73
3   335982         CL White           10
4   335982        DJ Hussey           12
5   335982        JH Kallis            7
6   335982       MV Boucher            9
7   335982  Mohammad Hafeez            3
8   335982          P Kumar           15
9   335982         R Dravid            3


In [14]:
# Step 3: Merge runs and balls faced into one table
batsman_stats = batsman_match_runs.merge(batsman_balls, on=['matchId', 'batsman'])

# Step 4: Calculate strike rate
batsman_stats['strike_rate'] = (batsman_stats['total_runs'] / batsman_stats['balls_faced']) * 100

print("Batsman stats with strike rate:")
print(batsman_stats.head(10))


Batsman stats with strike rate:
   matchId          batsman  total_runs  balls_faced  strike_rate
0   335982        AA Noffke           9           10    90.000000
1   335982          B Akhil           0            2     0.000000
2   335982      BB McCullum         158           73   216.438356
3   335982         CL White           6           10    60.000000
4   335982        DJ Hussey          12           12   100.000000
5   335982        JH Kallis           8            7   114.285714
6   335982       MV Boucher           7            9    77.777778
7   335982  Mohammad Hafeez           5            3   166.666667
8   335982          P Kumar          18           15   120.000000
9   335982         R Dravid           2            3    66.666667


In [15]:
# Step 5: Calculate wickets taken by each bowler per match
# Only count actual dismissals, not run outs (run out is not credited to bowler)
wickets = deliveries[~deliveries['dismissal_kind'].isin(['run out', 'none', 'retired hurt'])]

bowler_wickets = wickets.groupby(['matchId', 'bowler'])['dismissal_kind'].count().reset_index()
bowler_wickets.columns = ['matchId', 'bowler', 'wickets']

print("Bowler wickets:")
print(bowler_wickets.head(10))

Bowler wickets:
   matchId      bowler  wickets
0   335982   AA Noffke        1
1   335982  AB Agarkar        3
2   335982    AB Dinda        2
3   335982    I Sharma        1
4   335982   JH Kallis        1
5   335982   LR Shukla        1
6   335982  SC Ganguly        2
7   335982      Z Khan        1
8   335983       B Lee        1
9   335983   IK Pathan        2


In [16]:
# Step 6: Calculate economy rate for each bowler per match
# Economy = (runs conceded / balls bowled) * 6
# Only count legal balls (no wides, no no balls)
bowler_runs = deliveries[deliveries['isWide'] == 0].groupby(
    ['matchId', 'bowler'])['batsman_runs'].sum().reset_index()
bowler_runs.columns = ['matchId', 'bowler', 'runs_conceded']

bowler_balls = deliveries[deliveries['isWide'] == 0].groupby(
    ['matchId', 'bowler'])['ball'].count().reset_index()
bowler_balls.columns = ['matchId', 'bowler', 'balls_bowled']

# Merge runs and balls into one table
bowler_stats = bowler_runs.merge(bowler_balls, on=['matchId', 'bowler'])

# Calculate economy rate
bowler_stats['economy'] = (bowler_stats['runs_conceded'] / bowler_stats['balls_bowled']) * 6

print("Bowler stats with economy:")
print(bowler_stats.head(10))


Bowler stats with economy:
   matchId      bowler  runs_conceded  balls_bowled    economy
0   335982   AA Noffke             35            24   8.750000
1   335982  AB Agarkar             21            24   5.250000
2   335982    AB Dinda              7            18   2.333333
3   335982    CL White             22             6  22.000000
4   335982    I Sharma              6            18   2.000000
5   335982   JH Kallis             47            24  11.750000
6   335982   LR Shukla              9             7   7.714286
7   335982     P Kumar             37            24   9.250000
8   335982    SB Joshi             26            18   8.666667
9   335982  SC Ganguly             20            24   5.000000


In [17]:
# Step 7: Merge bowler wickets and economy into one table
bowler_full_stats = bowler_stats.merge(bowler_wickets, on=['matchId', 'bowler'], how='left')

# Fill NaN wickets with 0 (bowler bowled but took no wickets)
bowler_full_stats['wickets'] = bowler_full_stats['wickets'].fillna(0)

print("Full bowler stats:")
print(bowler_full_stats.head(10))
print("\nShape:", bowler_full_stats.shape)

Full bowler stats:
   matchId      bowler  runs_conceded  balls_bowled    economy  wickets
0   335982   AA Noffke             35            24   8.750000      1.0
1   335982  AB Agarkar             21            24   5.250000      3.0
2   335982    AB Dinda              7            18   2.333333      2.0
3   335982    CL White             22             6  22.000000      0.0
4   335982    I Sharma              6            18   2.000000      1.0
5   335982   JH Kallis             47            24  11.750000      1.0
6   335982   LR Shukla              9             7   7.714286      1.0
7   335982     P Kumar             37            24   9.250000      0.0
8   335982    SB Joshi             26            18   8.666667      0.0
9   335982  SC Ganguly             20            24   5.000000      2.0

Shape: (13845, 6)


In [18]:
# Step 8: Save the feature tables to data folder
batsman_stats.to_csv(r"C:\Users\dhruv\playXI\data\batsman_stats.csv", index=False)
bowler_full_stats.to_csv(r"C:\Users\dhruv\playXI\data\bowler_stats.csv", index=False)

print("Files saved successfully!")
print("batsman_stats.csv saved!")
print("bowler_stats.csv saved!")


Files saved successfully!
batsman_stats.csv saved!
bowler_stats.csv saved!
